# Simulations

This notebook covers both simulation backends in `qcal`:

- **`StateVectorSimulator`** — exact, noise-free simulation via pure state vectors
- **`DensityMatrixSimulator`** — mixed-state simulation supporting arbitrary noise channels

Both backends support qubit and qutrit circuits. The density-matrix simulator accepts noise via a dictionary of gate-name → channel mappings or via the structured `ErrorModel` classes.

In [ ]:
import numpy as np
import jax.numpy as jnp
import quax

import qcal as qc
from qcal.simulation import (
    DensityMatrixSimulator,
    StateVectorSimulator,
    AmplitudeDamping,
    CustomErrorModel,
    DephasingNoise,
    DepolarizingNoise,
    LeakageNoise,
    RelaxationNoise,
    RelaxationParams,
    SeepageNoise,
    UnitaryError,
)

%load_ext autoreload
%autoreload 2

## 1. StateVectorSimulator

The `StateVectorSimulator` evolves a pure state vector exactly. It supports both exact probability mode (`n_shots=None`, the default) and shot-based sampling (`n_shots=N`).

### Qubit simulation

In [64]:
# Bell state |Φ+⟩ = (|00⟩ + |11⟩) / √2
circuit = qc.Circuit([
    qc.Cycle({qc.H(0)}),
    qc.Cycle({qc.CNOT(0, 1)}),
])
circuit.measure()
circuit.draw()

In [65]:
sim = StateVectorSimulator()
sim.run(circuit)
circuit.results  # expect {'00': 0.5, '11': 0.5}

,00,11
counts,0.5,0.5
probabilities,0.5,0.5


In [66]:
sim.states[0].pretty_print()

'0.707|00⟩ + 0.707|11⟩'

In [67]:
# Shot-based sampling
sim.run(circuit, n_shots=1000)
circuit.results  # stochastic; expect ~500 '00', ~500 '11'

,00,11
counts,505,495
probabilities,0.505,0.495


### Qubit simulation with errors

The `StateVectorSimulator` can simulate the outcome for gates with unitary (or coherent errors), but not stochastic (probabilistic) errors (for that, you must use the `DensitMatrixSimulator`). Do do this, you can re-write the unitary matrix for a gate and create a circuit using the erroneous gates:

In [30]:
import numpy as np

def H_overrot(eps):
    d = np.pi*eps/2
    Hm = np.array([[1,1],[1,-1]])/np.sqrt(2)
    return np.cos(d)*Hm - 1j*np.sin(d)*np.eye(2)

def cnot_overrot(eps):
    d = np.pi*eps/2
    blk = np.cos(d)*np.array([[0,1],[1,0]]) - 1j*np.sin(d)*np.eye(2)
    U = np.eye(4, dtype=complex); U[2:,2:] = blk
    return U

In [45]:
H0 = qc.H(0)
H0.unitary = H_overrot(eps=0.01)

CNOT01 = qc.CNOT(0, 1)
CNOT01.unitary = cnot_overrot(eps=0.1)

In [46]:
# Bell state |Φ+⟩ = (|00⟩ + |11⟩) / √2
circuit = qc.Circuit([
    qc.Cycle({H0}),
    qc.Cycle({CNOT01}),
])
circuit.measure()
circuit.draw()

In [47]:
sim = StateVectorSimulator()
sim.run(circuit, n_shots=1000)
circuit.results

,00,10,11
counts,504,13,483
probabilities,0.504,0.013,0.483


In [48]:
sim.states[0].pretty_print()

'(0.707-0.016i)|00⟩ - 0.111i|10⟩ + 0.698|11⟩'

### Qutrit simulation

Qutrit gates live in `qcal.gates.single_qutrit`. The simulator infers the local Hilbert-space dimension (2 or 3) from the gate unitaries.

In [68]:
# Equal superposition of |0⟩, |1⟩, |2⟩ via qutrit Hadamard
c1 = qc.Circuit([qc.Cycle({qc.H3(0)})])
c1.measure()
c1.draw()

In [69]:
sim = StateVectorSimulator()
sim.run(c1)
c1.results  # expect {'0': 1/3, '1': 1/3, '2': 1/3}

,0,1,2
counts,0.333333,0.333333,0.333333
probabilities,0.333333,0.333333,0.333333


In [5]:
# X01 then X12 walks |0⟩ → |1⟩ → |2⟩
c2 = qc.Circuit([
    qc.Cycle({qc.X01(0)}),
    qc.Cycle({qc.X12(0)}),
])
sim.run(c2)
c2.results  # expect {'2': 1.0}

,2
counts,1.0
probabilities,1.0


In [6]:
# Partial rotation in EF subspace: |0⟩ → X→ |1⟩ → Rx12(π/2) → superposition
c3 = qc.Circuit([
    qc.Cycle({qc.X(0)}),
    qc.Cycle({qc.Rx12(0, np.pi / 2)}),
])
sim.run(c3)
c3.results  # expect {'1': 0.5, '2': 0.5}

,1,2
counts,0.5,0.5
probabilities,0.5,0.5


## 2. DensityMatrixSimulator

The `DensityMatrixSimulator` tracks the full density matrix ρ. A noise channel is applied after each gate unitary. Noise is specified via:

1. A **`dict`** mapping gate class names (strings) to `quax` channels — quick and flexible
2. An **`ErrorModel`** subclass — structured, per-category rates with optional per-gate channels and readout noise

### Ideal simulation (no noise)

With no noise model, `DensityMatrixSimulator` gives the same result as `StateVectorSimulator`.

In [7]:
circuit = qc.Circuit([
    qc.Cycle({qc.H(0)}),
    qc.Cycle({qc.CNOT(0, 1)}),
])
circuit.measure()

sim = DensityMatrixSimulator()
sim.run(circuit)
circuit.results  # expect {'00': 0.5, '11': 0.5}

,00,11
counts,0.5,0.5
probabilities,0.5,0.5


### Dict-based noise

Pass a `dict` of `gate_name → quax channel` for ad-hoc noise without creating a `NoiseModel` object. This is convenient when you have heterogeneous channels per gate type.

In [8]:
noise_model = {
    'H':    quax.channels.depolarizing(0.01),
    'CNOT': quax.channels.depolarizing(0.05),
}

sim = DensityMatrixSimulator(noise_model=noise_model)
sim.run(circuit, n_shots=1000)
circuit.results

,00,01,10,11
counts,466,29,34,471
probabilities,0.466,0.029,0.034,0.471


A dict value can be any `quax.KrausMap`. Here is a helper that builds a single-qubit Pauli channel from independent X/Y/Z error rates:

In [9]:
def pauli_channel(p_x: float, p_y: float, p_z: float) -> quax.KrausMap:
    p_i = 1.0 - p_x - p_y - p_z
    ops = jnp.stack([
        jnp.sqrt(p_i) * jnp.eye(2, dtype=complex),
        jnp.sqrt(p_x) * jnp.array([[0, 1], [1, 0]], dtype=complex),
        jnp.sqrt(p_y) * jnp.array([[0, -1j], [1j, 0]], dtype=complex),
        jnp.sqrt(p_z) * jnp.array([[1, 0], [0, -1]], dtype=complex),
    ])
    return quax.KrausMap.from_matrix(ops, dims=((2,), (2,)))

noise_model = {
    'H':    pauli_channel(p_x=0.01, p_y=0.005, p_z=0.01),
    'CNOT': pauli_channel(p_x=0.02, p_y=0.01,  p_z=0.02),
}

sim = DensityMatrixSimulator(noise_model=noise_model)
sim.run(circuit, n_shots=1000)
circuit.results

,00,01,10,11
counts,474,22,29,475
probabilities,0.474,0.022,0.029,0.475


## 3. Built-in error models

`ErrorModel` subclasses map broad gate categories (`single_qubit`, `two_qubit`, `single_qutrit`, `two_qutrit`) to channels, so one rate applies to all gates in that category.

In [4]:
circuit = qc.Circuit([
    qc.Cycle({qc.H(0)}),
    qc.Cycle({qc.CNOT(0, 1)}),
])
circuit.measure()
circuit.draw()

### DepolarizingNoise

In [15]:
noise = DepolarizingNoise(single_qubit=0.01, two_qubit=0.05)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

,00,01,10,11
counts,466,31,33,470
probabilities,0.466,0.031,0.033,0.47


### AmplitudeDamping

Models energy relaxation (T1 decay): the excited state decays to ground with probability γ.

In [14]:
noise = AmplitudeDamping(single_qubit=0.02, two_qubit=0.05)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

,00,01,10,11
counts,535,11,27,427
probabilities,0.535,0.011,0.027,0.427


### DephasingNoise

Models pure dephasing (T2 decay without energy relaxation).

In [16]:
noise = DephasingNoise(single_qubit=0.01, two_qubit=0.03)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

,00,11
counts,483,517
probabilities,0.483,0.517


### RelaxationNoise (T1 + Tφ)

`RelaxationNoise` combines energy relaxation (T1) and pure dephasing (Tφ) into a single channel via `RelaxationParams`. This is the most physically accurate model for superconducting qubits.

`t` is the gate duration; `tphi` is the *pure-dephasing* time — convert from T2 via `1/Tφ = 1/T2 − 1/(2·T1)`.

In [17]:
single_q_params = RelaxationParams(t1=50e-6, tphi=30e-6, t=50e-9)
two_q_params    = RelaxationParams(t1=35e-6, tphi=20e-6, t=200e-9)

noise = RelaxationNoise(
    single_qubit=single_q_params,
    two_qubit=two_q_params,
)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

,00,01,10,11
counts,516,1,3,480
probabilities,0.516,0.001,0.003,0.48


### Qutrit noise: LeakageNoise and SeepageNoise

`LeakageNoise` models population leaking from the computational subspace {|0⟩, |1⟩} into |2⟩. `SeepageNoise` is the reverse (|2⟩ → computational). Both accept `single_qutrit` and `two_qutrit` rates.

In [18]:
# Circuit in the qutrit space
qutrit_circuit = qc.Circuit([
    qc.Cycle({qc.X01(0)}),
    qc.Cycle({qc.X12(0)}),
])
qutrit_circuit.measure()
qutrit_circuit.draw()

In [23]:
# Leakage out of the computational subspace
noise = LeakageNoise(single_qutrit=0.05)
sim = DensityMatrixSimulator(noise_model=noise)
sim.run(qutrit_circuit, n_shots=1000)
qutrit_circuit.results

,1,2
counts,39,961
probabilities,0.039,0.961


In [24]:
# Seepage back into the computational subspace
noise = SeepageNoise(single_qutrit=0.02)
sim = DensityMatrixSimulator(noise_model=noise)
sim.run(qutrit_circuit, n_shots=1000)
qutrit_circuit.results

,1,2
counts,22,978
probabilities,0.022,0.978


### UnitaryError

`UnitaryError` models coherent (systematic) gate errors as ρ → U ρ U†, where U is a fixed unitary matrix. Unlike stochastic channels, this preserves state purity.

Two granularities are available:

* **Category-level** (*single_qubit*, *two_qubit*, etc.) — the same unitary error is applied after every gate in that category.
* **Per-gate-instance** (*gate_unitaries*) — a specific unitary for an individual gate instance (e.g. `CNOT((0, 1))`). The per-instance channel takes priority: if a gate appears in *gate_unitaries*, the category-level channel is skipped for that gate.

In [ ]:
# Category-level: small Rz over-rotation on every single-qubit gate
theta = 0.05  # radians
Rz = np.array([
    [np.exp(-0.5j * theta), 0],
    [0,                     np.exp(0.5j * theta)],
])

noise = UnitaryError(single_qubit=Rz)
sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results  # H gets Rz over-rotation; slight leakage to |01⟩, |10⟩

Per-gate-instance channels override the category-level channel for that specific gate. Here `CNOT(0, 1)` gets its own (larger) coherent error while `H(0)` continues to use the category-level `Rz`.

In [ ]:
# Per-gate-instance: CNOT(0,1) gets a larger over-rotation (no category fallback)
theta_cnot = 0.2
Rz_cnot = np.array([
    [np.exp(-0.5j * theta_cnot), 0],
    [0,                          np.exp(0.5j * theta_cnot)],
])

noise = UnitaryError(
    single_qubit=Rz,                          # applied to H(0)
    gate_unitaries={qc.CNOT(0, 1): Rz_cnot}, # replaces category for CNOT(0,1)
)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

## 4. CustomErrorModel

`CustomErrorModel` composes multiple `ErrorModel` instances. Channels from each sub-model are applied sequentially after every gate.

In [5]:
circuit = qc.Circuit([
    qc.Cycle({qc.H(0)}),
    qc.Cycle({qc.CNOT(0, 1)}),
])
circuit.measure()
circuit.draw()

### Combining noise models

In [ ]:
noise = CustomErrorModel(
    DepolarizingNoise(single_qubit=0.005, two_qubit=0.02),
    RelaxationNoise(
        single_qubit=RelaxationParams(t1=50e-6, tphi=30e-6, t=50e-9),
        two_qubit=RelaxationParams(t1=35e-6, tphi=20e-6, t=200e-9),
    ),
)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

### Per-gate PTM channels (`from_PTM`)

`CustomErrorModel.from_PTM` accepts a `{gate_instance: PTM}` dict. The PTM convention is `R[i, j] = Tr(Pᵢ E(Pⱼ)) / d` where `Pₖ` are the (unnormalized) Pauli operators and `E` is the error channel. For a noise-free gate the PTM is the identity matrix.

In [ ]:
# Small coherent Z-rotation error on X(0): PTM with decayed off-diagonal elements
ptm_x0 = np.diag([1.0, 0.99, 0.99, 1.0])   # slight X-Y dephasing
ptm_cnot = np.diag([1.0, 0.98, 0.98, 0.98, 0.98, 0.98, 0.98, 0.98,
                    0.98, 0.98, 0.98, 0.98, 0.98, 0.98, 0.98, 0.98])

noise = CustomErrorModel.from_PTM({
    qc.X(0):       ptm_x0,
    qc.CNOT(0, 1): ptm_cnot,
})

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

### Per-gate Pauli noise (`from_PNR`)

`CustomErrorModel.from_PNR` accepts a `{gate_instance: {Pauli_string: probability}}` dict — the natural output format of Pauli Noise Reconstruction (PNR) / cycle benchmarking analysis. The identity Pauli is added implicitly with any remaining probability.

Single-qubit errors use 1-character strings (`'X'`, `'Z'`); two-qubit errors use 2-character strings (`'ZI'`, `'IZ'`, `'ZZ'`).

In [ ]:
noise = CustomErrorModel.from_PNR({
    qc.X(0):       {'Z': 0.005, 'X': 0.002},
    qc.CNOT(0, 1): {'ZI': 0.01, 'IZ': 0.01, 'ZZ': 0.002},
})

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

In [ ]:
# Combine PNR-based per-gate errors with a broadband depolarizing floor
noise = CustomErrorModel(
    DepolarizingNoise(single_qubit=0.003, two_qubit=0.015),
    CustomErrorModel.from_PNR({
        qc.X(0):       {'Z': 0.005},
        qc.CNOT(0, 1): {'ZZ': 0.002},
    }),
)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit, n_shots=1000)
circuit.results

## 5. Readout noise

Confusion matrices can be added to any `NoiseModel` via `add_readout_noise`. The matrix must use the **qcal row-stochastic convention**: `C[prep, meas] = P(measure meas | prepared prep)`. It is transposed internally to the quax column-stochastic convention.

For terminal `Meas` gates the confusion is applied classically to the probability vector. For `MCM` gates a noisy quantum instrument is applied mid-circuit.

In [13]:
# 95% readout fidelity on both qubits
cmat = np.array([[0.95, 0.05],   # prepared |0⟩: 95% measured as 0, 5% as 1
                 [0.05, 0.95]])  # prepared |1⟩: 5% measured as 0, 95% as 1

noise = DepolarizingNoise(single_qubit=0.01, two_qubit=0.05)
noise.add_readout_noise('Q0', cmat)
noise.add_readout_noise('Q1', cmat)

# Prepare |11⟩ and measure with readout confusion
circuit_xx = qc.Circuit([
    qc.Cycle({qc.X(0), qc.X(1)}),
])
circuit_xx.measure()

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(circuit_xx, n_shots=1000)
circuit_xx.results  # expect mostly '11', with ~10% leaked to '01'/'10'

,00,01,10,11
counts,3,48,53,896
probabilities,0.003,0.048,0.053,0.896


In [ ]:
# If you have a ReadoutFidelity.cmat DataFrame, pass the per-qubit slice directly.
# The DataFrame is also row-stochastic, so the same add_readout_noise call works:
#
#   ro = ReadoutFidelity(CustomQPU, config, [0, 1])
#   ro.run()
#   noise.add_readout_noise('Q0', ro.cmat['Q0'])
#   noise.add_readout_noise('Q1', ro.cmat['Q1'])

## 6. Mid-circuit measurements (MCM)

`MCM` gates apply a projective measurement mid-circuit and update the density matrix to the post-measurement state (averaged over outcomes). The measurement outcome is recorded as the leading bit(s) in the result bitstring, in sorted qudit order.

**Note:** `n_shots` must be specified when the circuit contains `MCM` gates; exact probability mode is not supported for mid-circuit measurements.

In [24]:
# X prepares |1⟩, MCM measures (outcome 1), X flips back to |0⟩, Meas reads |0⟩
# Bitstring order: MCM outcome first, then Meas outcome → '10'
mcm_circuit = qc.Circuit([
    qc.Cycle({qc.X(0)}),
    qc.Cycle({qc.MCM(0)}),
    qc.Cycle({qc.X(0)}),
    qc.Cycle({qc.Meas(0)}),
])

sim = DensityMatrixSimulator()
sim.run(mcm_circuit, n_shots=1000)

print(mcm_circuit.mcm_results[0])
print(mcm_circuit.results)

                  1
counts         1000
probabilities   1.0
                  0
counts         1000
probabilities   1.0


/opt/anaconda3/envs/qcal-package-test/lib/python3.11/site-packages/quax/channels.py:155: UserWarning: Explicitly requested dtype complex128 requested in zeros is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  K = jnp.zeros((d_total, d_total), dtype=jnp.complex128)


In [26]:
# MCM with readout confusion: noisy instrument applied at measurement
cmat = np.array([[0.95, 0.05],
                 [0.05, 0.95]])

noise = DepolarizingNoise(single_qubit=0.01)
noise.add_readout_noise('Q0', cmat)

sim = DensityMatrixSimulator(noise_model=noise)
sim.run(mcm_circuit, n_shots=1000)

print(mcm_circuit.mcm_results[0])
print(mcm_circuit.results)

                   0      1
counts             5    995
probabilities  0.005  0.995
                   0      1
counts           934     66
probabilities  0.934  0.066


/opt/anaconda3/envs/qcal-package-test/lib/python3.11/site-packages/quax/channels.py:155: UserWarning: Explicitly requested dtype complex128 requested in zeros is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  K = jnp.zeros((d_total, d_total), dtype=jnp.complex128)
